<a href="https://colab.research.google.com/github/jkshakthi999/AI-Image-Colorization/blob/main/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

!pip install -q gradio scikit-image
print("Libraries installed.")

PyTorch: 2.14.1+cpu
GPU available: False
Libraries installed.


In [ ]:
import os, io, glob, time, shutil, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms as T
from PIL import Image, ImageOps
from skimage import color
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/era_colorization"      # photos + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage - files will be lost when the session ends.")
    DATA_ROOT = "/content/era_colorization"

# ------------------------- eras (folder names) -------------------------
# You may edit this list BEFORE the first training. If you change it later, train again.
PERIODS = ["1900s-1910s", "1920s", "1930s", "1940s", "1950s"]
P2I = {p: i for i, p in enumerate(PERIODS)}

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256        # training size (divisible by 32)
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
MIN_SAT = 0.03        # photos with lower average saturation are treated as black & white and skipped in training
BATCH_SIZE = 16
LR = 3e-4
COLOR_W = 10.0        # weight of the colour loss
CLS_W = 1.0           # weight of the era-classification loss
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
CACHE_ROOT = "/content/era_cache"                       # resized copies for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "era_colorizer.pt")       # trained model

for p in PERIODS:
    os.makedirs(os.path.join(DATA_ROOT, p), exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)
print("Eras:", PERIODS)

Mounted at /content/drive
Device: cpu
Data folder: /content/drive/MyDrive/era_colorization
Eras: ['1900s-1910s', '1920s', '1930s', '1940s', '1950s']


In [ ]:
def prepare_period(p):
    # reads photos of one era, skips bad / black & white ones, saves resized copies for fast training
    out_dir = os.path.join(CACHE_ROOT, p)
    os.makedirs(out_dir, exist_ok=True)
    good, bad = [], []
    for f in sorted(glob.glob(os.path.join(DATA_ROOT, p, "*"))):
        if not f.lower().endswith(EXTS):
            continue
        try:
            with Image.open(f) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((os.path.basename(f), "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((os.path.basename(f), "looks black & white (training photos must be colour)"))
                continue
            s = 320.0 / min(w, h)
            if s < 1.0:
                im = im.resize((max(1, int(round(w * s))), max(1, int(round(h * s)))), Image.LANCZOS)
            dst = os.path.join(out_dir, f"{p}_{len(good):05d}.jpg")
            im.save(dst, quality=95)
            good.append(dst)
        except Exception as e:
            bad.append((os.path.basename(f), f"unreadable: {e}"))
    return good, bad


class EraDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items            # list of (path, era_id)
        self.train = train
        if train:
            self.tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)), T.RandomHorizontalFlip()])
        else:
            self.tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, pid = self.items[i]
        with Image.open(path) as im:
            img = self.tf(im.convert("RGB"))
        lab = color.rgb2lab(np.asarray(img)).astype("float32")
        L = torch.from_numpy(lab[..., 0:1] / 50.0 - 1.0).permute(2, 0, 1)     # lightness -> [-1, 1]
        ab = torch.from_numpy(lab[..., 1:] / 128.0).permute(2, 0, 1)          # colour    -> ~[-1, 1]
        if self.train:
            # make the grayscale input look like old photographs: contrast / brightness changes and film grain
            L = L * random.uniform(0.75, 1.25) + random.uniform(-0.15, 0.15)
            if random.random() < 0.5:
                L = L + torch.randn_like(L) * random.uniform(0.01, 0.05)
            L = L.clamp(-1.0, 1.0)
        return L, ab, pid


class FiLM(nn.Module):
    # scales and shifts feature maps depending on the selected era
    def __init__(self, emb_dim, ch):
        super().__init__()
        self.fc = nn.Linear(emb_dim, ch * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        g, b = self.fc(e).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.film = FiLM(emb_dim, out_ch)

    def forward(self, x, skip, e):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.film(x, e)


class EraColorizer(nn.Module):
    # grayscale (L) + era  ->  colour (ab) ;  grayscale (L)  ->  era logits
    def __init__(self, n_periods, emb_dim=64, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.emb = nn.Embedding(n_periods, emb_dim)
        self.film4 = FiLM(emb_dim, 512)
        self.up3 = UpBlock(512, 256, 256, emb_dim)
        self.up2 = UpBlock(256, 128, 128, emb_dim)
        self.up1 = UpBlock(128, 64, 64, emb_dim)
        self.up0 = UpBlock(64, 64, 64, emb_dim)
        self.head = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 2, 1), nn.Tanh())
        # era classifier on the deepest encoder features
        self.cls = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(512, n_periods))

    def encode(self, L):
        x = L.repeat(1, 3, 1, 1)
        s0 = self.stem(x)
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.l3(s2)
        s4 = self.l4(s3)
        return s0, s1, s2, s3, s4

    def era_logits(self, L):
        return self.cls(self.encode(L)[4])

    def forward(self, L, period):
        s0, s1, s2, s3, s4 = self.encode(L)
        logits = self.cls(s4)                      # what era does this photo look like?
        e = self.emb(period)                       # which era's palette should be used?
        d = self.up3(self.film4(s4, e), s3, e)
        d = self.up2(d, s2, e)
        d = self.up1(d, s1, e)
        d = self.up0(d, s0, e)
        d = F.interpolate(d, size=L.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(d), logits                # ab in [-1, 1], era logits

print("Dataset and model code ready.")

Dataset and model code ready.


In [ ]:
model = None            # trained model (filled by training, or loaded from Drive in Cell 5)
ACTIVE_PERIODS = []     # eras the model was trained on


def count_photos(p):
    folder = os.path.join(DATA_ROOT, p)
    if not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.lower().endswith(EXTS)])


def dataset_status():
    rows = ["| Era | Photos saved in Drive |", "|---|---|"]
    for p in PERIODS:
        n = count_photos(p)
        flag = "" if n >= 20 else ("  (need more, 20+ recommended)" if n > 0 else "  (none yet)")
        rows.append(f"| {p} | {n}{flag} |")
    return "\n".join(rows)


def save_uploads(period, uploaded):
    if not uploaded:
        return "No photos selected. Click the upload box, choose photos, then click Save.", dataset_status(), None
    folder = os.path.join(DATA_ROOT, period)
    os.makedirs(folder, exist_ok=True)
    ok = bad = 0
    stamp = int(time.time())
    for i, f in enumerate(uploaded):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(folder, f"up_{stamp}_{i:04d}{os.path.splitext(path)[1].lower()}"))
        ok += 1
    msg = f"Saved **{ok}** photo(s) to **{period}**." + (f" Skipped {bad} (not valid images)." if bad else "")
    return msg + " Choose another era to upload more, or go to the Train Model tab.", dataset_status(), None


def train_model(epochs):
    global model, ACTIVE_PERIODS
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    items, counts = [], {}
    for p in PERIODS:
        good, bad = prepare_period(p)
        counts[p] = len(good)
        yield emit(f"  {p:12s} usable photos: {len(good):4d}   skipped: {len(bad)}")
        for name, why in bad[:3]:
            yield emit(f"      skipped {name}: {why}")
        if 0 < len(good) < 20:
            yield emit(f"    WARNING: only {len(good)} photos for {p}; results for this era will be weak.")
        items += [(f, P2I[p]) for f in good]
    if len(items) < 8:
        raise gr.Error("Fewer than 8 usable colour photos in total. Upload more colour photos in the first tab.")
    active = [p for p in PERIODS if counts[p] > 0]
    if len(active) < 2:
        yield emit("NOTE: only one era has photos, so era detection cannot learn to tell eras apart. Upload at least 2 eras.")
    yield emit("Eras that will be learned: " + ", ".join(active))

    random.shuffle(items)
    n_val = max(2, int(0.1 * len(items)))
    val_items, train_items = items[:n_val], items[n_val:]
    train_ds = EraDataset(train_items, train=True)
    val_ds = EraDataset(val_items, train=False)
    per_era = {}
    for _, pid in train_items:
        per_era[pid] = per_era.get(pid, 0) + 1
    weights = [1.0 / per_era[pid] for _, pid in train_items]          # every era is sampled equally often
    bs = min(BATCH_SIZE, len(train_ds))
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    train_dl = DataLoader(train_ds, batch_size=bs, sampler=sampler, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training photos: {len(train_ds)} | validation photos: {len(val_ds)} | batch size: {bs}")

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = EraColorizer(len(PERIODS), pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    color_loss = nn.SmoothL1Loss()
    cls_loss = nn.CrossEntropyLoss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for L, ab, pid in train_dl:
            L, ab, pid = L.to(device), ab.to(device), pid.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred, logits = net(L, pid)
            loss = COLOR_W * color_loss(pred.float(), ab) + CLS_W * cls_loss(logits.float(), pid)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        v_col = v_cls = 0.0
        correct = total = 0
        with torch.no_grad():
            for L, ab, pid in val_dl:
                L, ab, pid = L.to(device), ab.to(device), pid.to(device)
                pred, logits = net(L, pid)
                v_col += color_loss(pred.float(), ab).item()
                v_cls += cls_loss(logits.float(), pid).item()
                correct += (logits.argmax(1) == pid).sum().item()
                total += pid.numel()
        v_col /= len(val_dl)
        v_cls /= len(val_dl)
        acc = correct / max(1, total)
        va = COLOR_W * v_col + CLS_W * v_cls

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict(), "periods": PERIODS, "active": active}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.3f}   val colour {v_col:.4f}   "
                   f"era accuracy {acc * 100:4.1f}%   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    # per-era validation results of the best model
    stats = {p: [0, 0, 0.0] for p in PERIODS}      # correct, total, colour error sum
    with torch.no_grad():
        for L, ab, pid in val_dl:
            pred, logits = net(L.to(device), pid.to(device))
            pred, pid_c = pred.float().cpu(), pid
            err = (pred - ab).abs().mean(dim=(1, 2, 3))
            guess = logits.argmax(1).cpu()
            for k in range(len(pid_c)):
                name = PERIODS[int(pid_c[k])]
                stats[name][1] += 1
                stats[name][0] += int(guess[k] == pid_c[k])
                stats[name][2] += float(err[k])
    yield emit("\nPer-era results on validation photos (era detection accuracy | colour error, lower is better):")
    for p in PERIODS:
        c, n, e = stats[p]
        if n > 0:
            yield emit(f"  {p:12s} {100.0 * c / n:5.1f}%   {e / n:.4f}   ({n} photos)")
    yield emit("Note: era detection from grayscale photos is hard. If accuracy is low, add more varied photos per era "
               "(different places, people, photographers) and use the manual era option.")

    model = net
    ACTIVE_PERIODS = active
    yield emit(f"\nDONE. Model saved to Drive. Go to the 'Colorize' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [ ]:
def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


def _prep(img):
    # PIL image -> model input tensor + full-resolution lightness
    img = ImageOps.exif_transpose(img).convert("RGB")
    W, H = img.size
    if max(W, H) > MAX_SIDE:
        s = MAX_SIDE / float(max(W, H))
        W, H = max(1, int(W * s)), max(1, int(H * s))
        img = img.resize((W, H), Image.LANCZOS)
    L_full = color.rgb2lab(np.asarray(img))[..., 0]                    # 0..100 at full resolution
    s = INFER_SIDE / float(max(W, H))
    small = img.resize((_round32(W * s), _round32(H * s)), Image.BICUBIC)
    L = color.rgb2lab(np.asarray(small))[..., 0] / 50.0 - 1.0
    x = torch.from_numpy(L).float()[None, None].to(device)
    return x, L_full, H, W


def _check_ready():
    if model is None or not ACTIVE_PERIODS:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")


@torch.no_grad()
def detect_era(img):
    # returns {era: probability} over the eras the model was trained on
    _check_ready()
    model.eval()
    x, _, _, _ = _prep(img)
    lg = (model.era_logits(x) + model.era_logits(torch.flip(x, dims=[3]))) / 2.0     # flip test-time augmentation
    lg = lg.float().cpu()[0]
    mask = torch.full_like(lg, -1e9)
    for p in ACTIVE_PERIODS:
        mask[P2I[p]] = 0.0
    probs = torch.softmax(lg + mask, dim=0).numpy()
    return {p: float(probs[P2I[p]]) for p in ACTIVE_PERIODS}


@torch.no_grad()
def colorize(img, period, saturation=1.0):
    # colorizes with the palette of the given era; keeps the original brightness (Lab colour space)
    _check_ready()
    if period not in ACTIVE_PERIODS:
        raise gr.Error(f"The model has no training photos for '{period}'. Available: {', '.join(ACTIVE_PERIODS)}")
    model.eval()
    x, L_full, H, W = _prep(img)
    pid = torch.tensor([P2I[period]], device=device)
    ab = model(x, pid)[0].float().cpu()
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)[0]
    ab = (ab.permute(1, 2, 0).numpy() * 128.0 * saturation).clip(-128, 127)
    lab = np.concatenate([L_full[..., None], ab], axis=-1)
    rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        if ck["periods"] == PERIODS:
            model = EraColorizer(len(PERIODS), pretrained=False).to(device)
            model.load_state_dict(ck["model"])
            model.eval()
            ACTIVE_PERIODS = ck.get("active", PERIODS)
            print("Loaded saved model from Drive. Eras:", ACTIVE_PERIODS)
        else:
            print("Saved model uses a different era list; please train again in the app.")
    except Exception as e:
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload photos and train inside the app (next cell).")

No saved model yet. Upload photos and train inside the app (next cell).


In [ ]:
try:
    demo.close()          # close an older app if this cell is re-run
except Exception:
    pass

AUTO = "auto"


def refresh_state():
    # shows the model status and the era choices: Auto-detect + the eras the model was trained on
    global model, ACTIVE_PERIODS
    if model is None and os.path.exists(CKPT):
        try:
            ck = torch.load(CKPT, map_location=device)
            if ck["periods"] == PERIODS:
                m = EraColorizer(len(PERIODS), pretrained=False).to(device)
                m.load_state_dict(ck["model"])
                m.eval()
                model = m
                ACTIVE_PERIODS = ck.get("active", PERIODS)
        except Exception as e:
            print("Could not load saved model:", e)
    if model is None or not ACTIVE_PERIODS:
        return ("⚠️ **No trained model yet.** Go to tab **2. Train Model** and click **Start training** "
                "(upload colour photos in tab 1 first).",
                gr.Radio(choices=[("🔎 Auto-detect era (recommended)", AUTO)] + [(p, p) for p in PERIODS],
                         value=AUTO, label="Era"))
    return (f"✅ **Model ready.** Eras: {', '.join(ACTIVE_PERIODS)}",
            gr.Radio(choices=[("🔎 Auto-detect era (recommended)", AUTO)] + [(p, p) for p in ACTIVE_PERIODS],
                     value=AUTO, label="Era (auto-detected, or choose manually)"))


def run_colorize(img, era_choice, saturation):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    try:
        t0 = time.time()
        probs = detect_era(img)
        top = max(probs, key=probs.get)
        if era_choice == AUTO:
            used = top
            note = f"Auto-detected era: **{top}** ({probs[top] * 100:.0f}% confidence). Palette used: **{used}**."
            if probs[top] < 0.4:
                note += " ⚠️ Low confidence - consider choosing the era manually."
        else:
            used = era_choice
            note = (f"Manual era: **{used}**. (The model's own guess was {top}, {probs[top] * 100:.0f}%.)")
        res = colorize(img, used, saturation)
        orig = ImageOps.exif_transpose(img).convert("RGB")
        if orig.size != res.size:
            orig = orig.resize(res.size, Image.LANCZOS)
        return res, orig, res, probs, note + f"  ({time.time() - t0:.1f}s)"
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


def compare_all(img, saturation):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    try:
        probs = detect_era(img)
        return [(colorize(img, p, saturation), f"{p}  ({probs[p] * 100:.0f}% match)") for p in ACTIVE_PERIODS]
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🕰 Time-Based Historical Photo Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Detects the era</b> &bull; <b>Uses that era's colour palette</b> &bull; "
    "<b>Change the era manually</b> &bull; <b>Compare all eras</b></p>"
)

with gr.Blocks(title="Time-Based Historical Photo Colorizer") as demo:
    gr.HTML(HEADER)
    with gr.Tabs():

        # ---------------- Tab 1: upload training photos ----------------
        with gr.Tab("1. Upload Training Photos"):
            gr.Markdown("Upload **real colour photos** from one era at a time (autochromes, Kodachrome, colour scans). "
                        "Select the era, choose the photos, click **Save photos**. Repeat for each era. "
                        "The model learns each era's palette, and learns to recognize the era from the photo's look. "
                        "Use varied photos (different places, people, photographers). Aim for 100+ per era (20 minimum), "
                        "and at least 2 eras so era detection can work.")
            with gr.Row():
                with gr.Column():
                    up_period = gr.Radio(choices=PERIODS, value=PERIODS[0], label="🕰 Era of these photos")
                    up_files = gr.File(label="Choose colour photos", file_count="multiple", file_types=["image"])
                    up_btn = gr.Button("Save photos", variant="primary")
                    up_msg = gr.Markdown()
                with gr.Column():
                    ds_md = gr.Markdown(dataset_status())
                    ds_btn = gr.Button("Refresh counts")
            up_btn.click(save_uploads, inputs=[up_period, up_files], outputs=[up_msg, ds_md, up_files])
            ds_btn.click(dataset_status, outputs=ds_md)

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading photos. It takes roughly 5-20 minutes on a T4 GPU. "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)

        # ---------------- Tab 3: colorize ----------------
        with gr.Tab("3. Colorize"):
            model_md = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="📷 Upload B&W Photo", height=380)
                    era_dd = gr.Radio(choices=[("🔎 Auto-detect era (recommended)", AUTO)] + [(p, p) for p in PERIODS],
                                      value=AUTO, label="Era (auto-detected, or choose manually)")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    btn_all = gr.Button("Compare all eras")
                    refresh_btn = gr.Button("Refresh model status")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload a photo and click **Colorize**. Leave *Auto-detect* on, or pick an era to override it.")
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("Era Detection"):
                            out_probs = gr.Label(label="How much the photo matches each era", num_top_classes=len(PERIODS))
                        with gr.Tab("Compare All Eras"):
                            gallery = gr.Gallery(label="Same photo in every era", columns=3)

    train_btn.click(train_model, inputs=epochs_sl, outputs=log_box).then(refresh_state, None, [model_md, era_dd])
    refresh_btn.click(refresh_state, None, [model_md, era_dd])
    btn.click(run_colorize, inputs=[inp, era_dd, sat], outputs=[out_main, out_orig, out_col, out_probs, status])
    btn_all.click(compare_all, inputs=[inp, sat], outputs=gallery)
    demo.load(refresh_state, None, [model_md, era_dd])

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1d2664d7b1a5a80c5f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 106MB/s]
